
## Folder 05 — File 05: Quarantine Management


The workflow:

- Read Silver datasets.
- Identify invalid records.
- Attach quarantine reasons.
- Separate valid and invalid records.
- Store invalid records in a centralized Delta table.
- Generate a quarantine summary.
- Inspect quarantined records.



### Quarantine Architecture

```text
┌──────────────────────────────────────────────┐
│              SILVER DATA TABLES              │
│                                              │
│     Customers · Products · Orders            │
│     Order Items · Payments · Shipments       │
│     Inventory Snapshots                      │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│           DATA QUALITY VALIDATION            │
│                                              │
│        Valid Records / Invalid Records       │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│              QUARANTINE TABLES               │
│                                              │
│  ┌────────────────────────────────────────┐  │
│  │ quarantine_records                    │  │
│  │ Latest execution's invalid records    │  │
│  └────────────────────────────────────────┘  │
│                                              │
│  ┌────────────────────────────────────────┐  │
│  │ quarantine_history                     │  │
│  │ Historical invalid records             │  │
│  └────────────────────────────────────────┘  │
│                                              │
│  ┌────────────────────────────────────────┐  │
│  │ quarantine_summary                     │  │
│  │ Failure counts by dataset and reason   │  │
│  └────────────────────────────────────────┘  │
└──────────────────────────────────────────────┘
```


In [0]:

## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SILVER_SCHEMA = "ecommerce_silver"

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_records"
)

QUARANTINE_HISTORY_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_history"
)

VALIDATION_SUMMARY_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_summary"
)

DATASETS = {
    "customers": "silver_customers",
    "products": "silver_products",
    "orders": "silver_orders",
    "order_items": "silver_order_items",
    "payments": "silver_payments",
    "shipments": "silver_shipments",
    "inventory": "silver_inventory"
}

RUN_TIMESTAMP = datetime.now()


## 3. Create quarantine schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}"
)

print("Silver schema is available.")


## 4. Define reusable quarantine function

def identify_invalid_records(
    df,
    dataset_name,
    validation_rules
):

    invalid_conditions = []

    for rule_name, condition in validation_rules:

        invalid_conditions.append(
            F.when(
                condition,
                F.lit(rule_name)
            )
        )

    result_df = (
        df
        .withColumn(
            "_quarantine_reason",
            F.concat_ws(
                " | ",
                *invalid_conditions
            )
        )
    )

    invalid_df = (
        result_df
        .filter(
            F.col("_quarantine_reason") != ""
        )
        .withColumn(
            "_quarantine_dataset",
            F.lit(dataset_name)
        )
        .withColumn(
            "_quarantined_at",
            F.current_timestamp()
        )
        .withColumn(
            "_quarantine_run_timestamp",
            F.lit(RUN_TIMESTAMP).cast("timestamp")
        )
    )

    valid_df = (
        result_df
        .filter(
            F.col("_quarantine_reason") == ""
        )
        .drop("_quarantine_reason")
    )

    return valid_df, invalid_df


## 5. Define dataset validation rules

VALIDATION_RULES = {

    "customers": [

        (
            "MISSING_CUSTOMER_ID",
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == "")
        ),

        (
            "MISSING_FIRST_NAME",
            F.col("first_name").isNull()
            | (F.trim(F.col("first_name")) == "")
        ),

        (
            "MISSING_EMAIL",
            F.col("email").isNull()
            | (F.trim(F.col("email")) == "")
        ),

        (
            "INVALID_EMAIL",
            F.col("email").isNotNull()
            & (
                ~F.col("email").rlike(
                    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
                )
            )
        ),

        (
            "MISSING_CREATED_DATE",
            F.col("created_at").isNull()
        )
    ],

    "products": [

        (
            "MISSING_PRODUCT_ID",
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == "")
        ),

        (
            "MISSING_PRODUCT_NAME",
            F.col("product_name").isNull()
            | (F.trim(F.col("product_name")) == "")
        ),

        (
            "INVALID_UNIT_PRICE",
            F.col("unit_price").isNull()
            | (F.col("unit_price") < 0)
        ),

        (
            "INVALID_UNIT_COST",
            F.col("unit_cost").isNull()
            | (F.col("unit_cost") < 0)
        ),

        (
            "MISSING_CREATED_DATE",
            F.col("created_at").isNull()
        )
    ],

    "orders": [

        (
            "MISSING_ORDER_ID",
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ),

        (
            "MISSING_CUSTOMER_ID",
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == "")
        ),

        (
            "MISSING_ORDER_TIMESTAMP",
            F.col("order_timestamp").isNull()
        ),

        (
            "INVALID_ORDER_STATUS",
            ~F.col("order_status").isin(
                "DELIVERED",
                "SHIPPED",
                "PROCESSING",
                "CANCELLED"
            )
        ),

        (
            "INVALID_ORDER_TOTAL",
            F.col("order_total").isNull()
            | (F.col("order_total") < 0)
        )
    ],

    "order_items": [

        (
            "MISSING_ORDER_ITEM_ID",
            F.col("order_item_id").isNull()
            | (F.trim(F.col("order_item_id")) == "")
        ),

        (
            "MISSING_ORDER_ID",
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ),

        (
            "MISSING_PRODUCT_ID",
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == "")
        ),

        (
            "INVALID_QUANTITY",
            F.col("quantity").isNull()
            | (F.col("quantity") <= 0)
        ),

        (
            "INVALID_UNIT_PRICE",
            F.col("unit_price").isNull()
            | (F.col("unit_price") < 0)
        ),

        (
            "INVALID_DISCOUNT",
            F.col("discount_rate").isNull()
            | (F.col("discount_rate") < 0)
            | (F.col("discount_rate") > 1)
        ),

        (
            "INVALID_LINE_TOTAL",
            F.col("line_total").isNull()
            | (F.col("line_total") < 0)
        )
    ],

    "payments": [

        (
            "MISSING_PAYMENT_ID",
            F.col("payment_id").isNull()
            | (F.trim(F.col("payment_id")) == "")
        ),

        (
            "MISSING_ORDER_ID",
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ),

        (
            "INVALID_PAYMENT_STATUS",
            ~F.col("payment_status").isin(
                "SUCCESS",
                "PENDING",
                "FAILED",
                "REFUNDED"
            )
        ),

        (
            "INVALID_PAYMENT_AMOUNT",
            F.col("amount").isNull()
            | (F.col("amount") < 0)
        ),

        (
            "MISSING_PAYMENT_TIMESTAMP",
            F.col("payment_timestamp").isNull()
        )
    ],

    "shipments": [

        (
            "MISSING_SHIPMENT_ID",
            F.col("shipment_id").isNull()
            | (F.trim(F.col("shipment_id")) == "")
        ),

        (
            "MISSING_ORDER_ID",
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ),

        (
            "INVALID_SHIPMENT_STATUS",
            ~F.col("shipment_status").isin(
                "PENDING",
                "PROCESSING",
                "SHIPPED",
                "IN_TRANSIT",
                "OUT_FOR_DELIVERY",
                "DELIVERED",
                "CANCELLED",
                "RETURNED"
            )
        ),

        (
            "DELIVERED_WITHOUT_DELIVERY_DATE",
            (
                F.col("shipment_status") == "DELIVERED"
            )
            & F.col("delivered_date").isNull()
        ),

        (
            "DELIVERY_BEFORE_SHIPMENT",
            F.col("delivered_date").isNotNull()
            & F.col("shipped_date").isNotNull()
            & (
                F.col("delivered_date") < F.col("shipped_date")
            )
        )
    ],

    "inventory": [

        (
            "MISSING_INVENTORY_ID",
            F.col("inventory_snapshot_id").isNull()
            | (F.trim(F.col("inventory_snapshot_id")) == "")
        ),

        (
            "MISSING_PRODUCT_ID",
            F.col("product_id").isNull()
            | (F.trim(F.col("product_id")) == "")
        ),

        (
            "MISSING_WAREHOUSE_ID",
            F.col("warehouse_id").isNull()
            | (F.trim(F.col("warehouse_id")) == "")
        ),

        (
            "INVALID_INVENTORY_QUANTITY",
            F.col("quantity_on_hand").isNull()
            | (F.col("quantity_on_hand") < 0)
        ),

        (
            "MISSING_SNAPSHOT_DATE",
            F.col("snapshot_date").isNull()
        )
    ]
}


## 6. Process all Silver datasets

valid_datasets = {}
invalid_datasets = {}

for dataset_name, table_suffix in DATASETS.items():

    source_table = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_suffix}"
    )

    print(f"Processing: {source_table}")

    source_df = spark.table(source_table)

    valid_df, invalid_df = identify_invalid_records(
        source_df,
        dataset_name,
        VALIDATION_RULES[dataset_name]
    )

    valid_datasets[dataset_name] = valid_df
    invalid_datasets[dataset_name] = invalid_df

    print(
        f"Valid records: {valid_df.count()}"
    )

    print(
        f"Invalid records: {invalid_df.count()}"
    )

print("All datasets processed.")


## 7. Display quarantined records

for dataset_name, invalid_df in invalid_datasets.items():

    print(f"Quarantined records: {dataset_name}")

    display(
        invalid_df.select(
            "_quarantine_dataset",
            "_quarantine_reason",
            "_quarantined_at",
            *[
                column_name
                for column_name in invalid_df.columns
                if not column_name.startswith("_quarantine")
            ]
        )
    )


## 8. Combine invalid records into one table

invalid_dataframes = list(
    invalid_datasets.values()
)

if invalid_dataframes:

    combined_quarantine_df = invalid_dataframes[0]

    for next_df in invalid_dataframes[1:]:

        combined_quarantine_df = (
            combined_quarantine_df.unionByName(
                next_df,
                allowMissingColumns=True
            )
        )

else:

    raise Exception("No quarantine datasets were configured.")


## 9. Save current quarantine records

(
    combined_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Current quarantine table saved: {QUARANTINE_TABLE}")


## 10. Append quarantine records to history

if spark.catalog.tableExists(QUARANTINE_HISTORY_TABLE):

    (
        combined_quarantine_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(QUARANTINE_HISTORY_TABLE)
    )

else:

    (
        combined_quarantine_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(QUARANTINE_HISTORY_TABLE)
    )

print(f"Quarantine history saved: {QUARANTINE_HISTORY_TABLE}")


## 11. Generate quarantine summary

quarantine_summary_df = (
    combined_quarantine_df
    .groupBy(
        "_quarantine_dataset",
        "_quarantine_reason"
    )
    .agg(
        F.count("*").alias("quarantined_records")
    )
    .withColumn(
        "summary_generated_at",
        F.current_timestamp()
    )
    .orderBy(
        "_quarantine_dataset",
        F.desc("quarantined_records")
    )
)

display(quarantine_summary_df)


## 12. Save quarantine summary

(
    quarantine_summary_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(VALIDATION_SUMMARY_TABLE)
)

print(f"Summary saved: {VALIDATION_SUMMARY_TABLE}")


## 13. Dataset-level valid and invalid record counts

dataset_statistics = []

for dataset_name in DATASETS:

    valid_count = valid_datasets[dataset_name].count()

    invalid_count = invalid_datasets[dataset_name].count()

    total_count = valid_count + invalid_count

    dataset_statistics.append({

        "dataset_name": dataset_name,

        "total_records": total_count,

        "valid_records": valid_count,

        "invalid_records": invalid_count,

        "valid_percentage": (
            round(valid_count / total_count * 100, 2)
            if total_count > 0
            else 0.0
        ),

        "invalid_percentage": (
            round(invalid_count / total_count * 100, 2)
            if total_count > 0
            else 0.0
        )
    })

statistics_df = spark.createDataFrame(
    dataset_statistics
)

display(statistics_df)


## 14. Final quarantine validation

current_quarantine_count = (
    spark.table(QUARANTINE_TABLE).count()
)

history_count = (
    spark.table(QUARANTINE_HISTORY_TABLE).count()
)

print("=" * 60)
print("QUARANTINE MANAGEMENT SUMMARY")
print("=" * 60)

print(f"Current quarantine records : {current_quarantine_count}")

print(f"Historical quarantine records : {history_count}")

print(f"Current quarantine table : {QUARANTINE_TABLE}")

print(f"History table : {QUARANTINE_HISTORY_TABLE}")

print(f"Summary table : {VALIDATION_SUMMARY_TABLE}")

print("=" * 60)

display(
    spark.table(QUARANTINE_TABLE)
)



## Quarantine Management Tables:


| Table | Purpose |
|---|---|
| `quarantine_records` | Stores invalid records identified during the latest execution. |
| `quarantine_history` | Maintains invalid records from previous and current executions for historical tracking. |
| `quarantine_summary` | Stores failure counts grouped by dataset and failure reason. |


### Storage Location:


All three tables are stored in the Silver schema:

`workspace.ecommerce_silver`